# 第十二章 文件、路径与 JSON 持久化


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第十二章 文件、路径与 JSON 持久化”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## 本章场景

记账程序目前的数据只在内存里：重启内核就全部丢失。要让账本“记住”自己，需要三件事：**路径**（文件放在哪里）、**文件读写**（怎么写入和读出）、**数据格式**（用什么结构保存——JSON 是 Python 与 Web 通用的标准格式）。

本章实现记账助手的关键能力：**持久化**。学完后，账目数据将能保存到磁盘、重新启动后完整恢复。

1. 12.1 Path 与目录
2. 12.2 文件读写：with 与编码
3. 12.3 JSON 序列化与反序列化
4. 12.4 封装：load_data 与 save_data
5. 12.5 工具速查
6. 12.6 易错点提醒
7. 12.7 练习与作业
8. 12.8 小结
9. 12.9 拓展作业



## Python 的学习主线

概念与语法 → 最小可运行代码 → 修改一个输入 → 处理边界条件 → 封装成函数 → 独立完成一个小任务

先预测输出，再运行代码；然后只改一个变量，最后把示例改写成自己的问题。重点检查变量类型、条件分支和中间结果。


## 本模块练习方式

基础：补全或改写一小段代码；提高：组合两个语法知识点；挑战：处理空输入、错误类型或边界值。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 本章目标

学完本章，你将能够：

- **理解**：理解 pathlib 路径、JSON 序列化与持久化。
- **操作**：能用 pathlib/JSON 读写文件并把记录持久化到磁盘。
- **迁移**：能把记账数据保存为 JSON 并重新加载、恢复成可用的记录对象。


## 12.1 Path 与目录

**背景引入**：前几章的账目数据都在内存里——重启内核就全没了。真实程序必须把数据存到硬盘：存哪里、怎么表示路径、怎么确认目录存在。pathlib 的 Path 对象把路径当对象操作，跨平台且不易出错。（把路径想成一根能用 / 一节节拼起来的"地址绳"，换什么系统都认出同一条路。）

### 12.1.1 路径构造

pathlib.Path 用 / 组合路径：Path("finance_app") / "data" / "records.json"。跨平台自动处理分隔符。

### 12.1.2 目录创建

path.parent.mkdir(parents=True, exist_ok=True) 递归创建目录，已存在不报错。

> 本章示例统一使用 TemporaryDirectory 临时目录，避免测试污染真实文件系统。




**练一练 12.1**：用 Path 构造 data/records.json 路径（基于当前目录），打印它的文件名、后缀、父目录，并确认父目录是否存在。


In [ ]:
# 请在下方填写代码
from pathlib import Path

# TODO：请在下方完成 —— 练一练 12.1：用 Path 构造 data/records.json 路径（基于当前目录），打印它的文件名、后缀、父


In [ ]:
from pathlib import Path

path = Path("data/records.json")
print(path.name)
print(path.suffix)
print(path.parent)
print(path.parent.exists())


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    root = Path(folder)
    data_dir = root / "finance_app" / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    data_file = data_dir / "records.json"
    print(data_file.name, data_file.suffix, data_file.parent.exists())


**输出解读**：路径名 records.json、扩展名 .json；data 目录已创建。with TemporaryDirectory 结束时自动清理。


## 12.2 文件读写：with 与编码

**背景引入**：路径知道了，接下来是真正的读写。核心习惯是 with open(...)：用完后自动关闭文件，忘记关闭是资源泄漏的常见原因。中文内容必须用 encoding="utf-8"，否则 Windows 默认编码会乱码。

### 12.2.1 with 语句

with path.open(...) as file: 打开文件，**无论是否出错**都会自动关闭——避免资源泄漏。

### 12.2.2 模式与编码

- "w"：覆盖写入（文件不存在则创建）；"a"：追加；"r"：读取（默认）；
- 中文文本必须显式指定 encoding="utf-8"。




**练一练 12.2**：把一行文本“2026-08-06|餐饮|35.5”写入 demo.txt（utf-8 编码），再用 with 读出来打印，最后删除该文件。


In [ ]:
# 请在下方填写代码
from pathlib import Path

# TODO：请在下方完成 —— 练一练 12.2：把一行文本“2026-08-06|餐饮|35.5”写入 demo.txt（utf-8 编码），再用 w


In [ ]:
from pathlib import Path

demo = Path("demo.txt")
with open(demo, "w", encoding="utf-8") as f:
    f.write("2026-08-06|餐饮|35.5\n")
with open(demo, "r", encoding="utf-8") as f:
    print(f.read().strip())
demo.unlink()


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    with path.open("w", encoding="utf-8") as file:
        file.write("餐饮：35.50 元\n")
    with path.open("r", encoding="utf-8") as file:
        content = file.read()
    print(repr(content))


**输出解读**：写入中文文本并读回；with 保证文件关闭；repr 显示读回的内容含换行符。


## 12.3 JSON 序列化与反序列化

**背景引入**：文本文件只能存字符串，但账目是字典、列表——怎么把结构化数据存下来？JSON 就是“Python 数据 ↔ 文本”的标准桥梁：json.dump 存盘、json.load 读回，字典列表原样还原。

### 12.3.1 文件级：dump 与 load

- json.dump(数据, 文件)：写入文件；
- json.load(文件)：从文件读取。

保存中文用 ensure_ascii=False，输出可读；indent=2 美化格式。

### 12.3.2 字符串级：dumps 与 loads

- json.dumps(数据) 返回字符串（网络传输、显示）；
- json.loads(字符串) 解析字符串。

> **易错点**：dump/load 操作**文件对象**，dumps/loads 操作**字符串**——别少写一个 s。




**练一练 12.3**：把账目列表存为 JSON 文件（ensure_ascii=False 保留中文），读回后打印；确认读回的数据类型是 list。


In [ ]:
# 请在下方填写代码
import json
from pathlib import Path

records = [{"date": "2026-08-06", "category": "餐饮", "amount": 35.5}]
# TODO：请在下方完成 —— 练一练 12.3：把账目列表存为 JSON 文件（ensure_ascii=False 保留中文），读回后打印；确认读回


In [ ]:
import json
from pathlib import Path

records = [{"date": "2026-08-06", "category": "餐饮", "amount": 35.5}]
path = Path("records_tmp.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False)
with open(path, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded)
print(type(loaded))
path.unlink()


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

records = [
    {"date": "2026-08-06", "type": "支出", "category": "餐饮", "amount": 35.5}
]
with TemporaryDirectory() as folder:
    path = Path(folder) / "records.json"
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)
    with path.open(encoding="utf-8") as file:
        loaded = json.load(file)
    print(path.read_text(encoding="utf-8"))
    print("保存前后相同：", loaded == records)


**输出解读**：账目列表写入 JSON 文件并读回；load 结果与原始数据完全一致。


## 12.4 封装：load_data 与 save_data

**背景引入**：读写代码不该散落在各处——把“读账本”“存账本”封装成两个函数，全程序统一调用。这是把前面学的函数与本章文件操作结合起来的关键一步。

把保存与读取封装成函数，供程序任意位置复用：

- save_data：自动创建目录，写入 JSON；
- load_data：文件不存在返回空列表（首次使用）。




**练一练 12.4**：封装 load_data(path) 与 save_data(path, data)：save 把数据写入 JSON，load 读回；用一条账目做“存 → 读 → 删”闭环。


In [ ]:
# 请在下方填写代码
import json
from pathlib import Path

# TODO：请在下方完成 —— 练一练 12.4：封装 load_data(path) 与 save_data(path, data)：save 把数据


In [ ]:
import json
from pathlib import Path


def save_data(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)


def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


path = Path("ledger_tmp.json")
save_data(path, [{"date": "2026-08-06", "amount": 35.5}])
print(load_data(path))
loaded_check = load_data(path)


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory


def save_data(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)


def load_data(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as file:
        return json.load(file)


with TemporaryDirectory() as folder:
    test_path = Path(folder) / "data" / "records.json"
    save_data([{"amount": 35.5}], test_path)
    print(load_data(test_path))


**输出解读**：save_data 创建 data 目录并写入；load_data 读回相同内容——这就是账本的持久化闭环。


## 12.5 工具速查

| 工具 | 用途 | 示例 |
| --- | --- | --- |
| Path(路径) | 构造路径对象 | Path("finance_app") |
| 路径 / 子路径 | 组合路径 | root / "data" |
| .name / .stem / .suffix | 文件名部件 | path.name |
| .parent | 上级目录 | path.parent |
| .mkdir(parents, exist_ok) | 创建目录 | data_dir.mkdir(parents=True, exist_ok=True) |
| .exists() / .is_file() / .is_dir() | 检查 | path.exists() |
| .glob("*.json") | 匹配文件 | data_dir.glob("*.json") |
| path.open(模式, encoding) | 打开文件 | path.open("w", encoding="utf-8") |
| file.write / read / readline / readlines | 读写文本 | 见 11.10 |
| json.dump / load | 文件级 JSON | 见 11.7 |
| json.dumps / loads | 字符串级 JSON | 见 11.7 |
| TemporaryDirectory | 临时目录 | with TemporaryDirectory() as folder |




## 12.6 本章实训：记账文件的持久化闭环

本节 3 个实验把「定位 → 读写 → 封装」串成完整闭环：先用 Path 定位账目文件，再用 with 读写文本，最后用 JSON 保存与读取记账记录。


### 实验 1：用 Path 定位账目文件

**操作步骤**：运行下方代码；把 data_dir 换成不存在的目录再运行一次，观察差异。

**观察要点**：Path 用 / 拼接路径；写文件前先 mkdir(exist_ok=True)。


In [ ]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
target = data_dir / "records.json"
print(target.name, target.suffix, target.parent)
print("文件存在：", target.exists())


### 实验 2：with 读写与 JSON 往返

**操作步骤**：运行下方代码，观察 json.dump 写入的文件内容。

**观察要点**：with 自动关闭文件；dump 写文件、dumps 返回字符串。


In [ ]:
import json

records = [{"日期": "2026-08-16", "分类": "餐饮", "金额": 35.5}]
with open("data/records.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

with open("data/records.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded)


### 实验 3：save_data / load_data 封装闭环

**操作步骤**：运行下方代码，再给 records 增加一条记录后重新保存、重新加载。

**观察要点**：读写逻辑封装成函数后，主流程只剩三行。


In [ ]:
import json
from pathlib import Path


def save_data(path, data):
    path = Path(path)
    path.parent.mkdir(exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


records = [{"日期": "2026-08-16", "分类": "餐饮", "金额": 35.5}]
save_data("data/records.json", records)
print(load_data("data/records.json"))


## 12.7 易错点提醒




### 12.7.1 Path 的 name、suffix、parent 与 with_suffix

路径部件操作返回**新对象**，原路径不变。with_suffix 用于生成备份路径等变体。




In [ ]:
from pathlib import Path

path = Path("finance_app/data/records.json")
print("文件名：", path.name)
print("主干名：", path.stem)
print("扩展名：", path.suffix)
print("上级目录：", path.parent)
print("备份路径：", path.with_suffix(".backup.json"))
print("原路径没变：", path)


**输出解读**：name/stem/suffix/parent 提取路径部件；with_suffix 生成新路径；原路径对象不变。


### 12.7.2 exists、is_file、is_dir 和 glob

读写前先检查路径；glob 按模式列出目录中的文件（返回生成器，sorted 后使用）。




In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    root = Path(folder)
    data_dir = root / "data"
    data_dir.mkdir()
    (data_dir / "records.json").write_text("[]", encoding="utf-8")
    (data_dir / "settings.json").write_text("{}", encoding="utf-8")
    print(data_dir.exists(), data_dir.is_dir())
    print((data_dir / "records.json").is_file())
    print(sorted(path.name for path in data_dir.glob("*.json")))


**输出解读**：exists/is_dir/is_file 检查路径性质；glob 匹配目录下所有 JSON 文件。


### 12.7.3 r、w、a 模式和 read 系列方法

- read() 全部内容；readline() 一行；readlines() 全部行列表；
- "w" 覆盖写入、"a" 追加写入。




In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "notes.txt"
    with path.open("w", encoding="utf-8") as file:
        file.write("午餐\n")
        file.write("地铁\n")
    with path.open("a", encoding="utf-8") as file:
        file.write("购物\n")
    with path.open("r", encoding="utf-8") as file:
        first_line = file.readline().strip()
        remaining = [line.strip() for line in file.readlines()]
    print(first_line)
    print(remaining)


**输出解读**：w 写入两行、a 追加一行、r 读取：第一行与剩余行分别处理。


### 12.7.4 dump/load 和 dumps/loads 别少一个 s

dumps/loads 处理字符串（不写文件）；dump/load 处理文件对象。记法：**带 s 的是字符串**。




In [ ]:
import json

records = [{"date": "2026-08-06", "type": "支出", "amount": 35.5}]
json_text = json.dumps(records, ensure_ascii=False, indent=2)
restored = json.loads(json_text)
print(json_text)
print(restored)
print(type(restored).__name__)
print(restored == records)


**输出解读**：dumps 生成 JSON 字符串，loads 解析回 Python 列表——往返一致。


### 12.7.5 os.path 和 Path 怎么对应

旧代码常见 os.path.join/basename/splitext；新代码统一用 pathlib（name/stem/suffix）。两者等价，pathlib 更简洁。




In [ ]:
import os
from pathlib import Path

old_style = os.path.join("finance_app", "data", "records.json")
new_style = Path("finance_app") / "data" / "records.json"
print(old_style)
print(new_style)
print(os.path.basename(old_style))
print(os.path.splitext(old_style))
print(new_style.name, new_style.stem, new_style.suffix)


**输出解读**：os.path 与 pathlib 表达同一路径；pathlib 的部件访问更直观。


## 12.8 练习与作业

练习分为基础、提高、挑战三级。先独立完成，再对照参考答案。




### 基础 1：路径构造与部件

构造 finance_app/data/records.json 路径，输出文件名、主干名、扩展名与上级目录。


In [ ]:
# 请在下方填写代码
from pathlib import Path

# TODO：请在下方完成 —— 基础 1：路径构造与部件 构造 finance_app/data/records.json 路径，输出文件名、主干名、扩


In [ ]:
from pathlib import Path

path = Path("finance_app") / "data" / "records.json"
print(path.name, path.stem, path.suffix, path.parent)


### 基础 2：文本文件读写

在临时目录中写入一行中文文本，再读取并输出。


In [ ]:
# 请在下方填写代码
from pathlib import Path
from tempfile import TemporaryDirectory

# TODO：请在下方完成 —— 基础 2：文本文件读写 在临时目录中写入一行中文文本，再读取并输出。


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    with path.open("w", encoding="utf-8") as file:
        file.write("餐饮：35.50 元\n")
    with path.open("r", encoding="utf-8") as file:
        print(file.read())


### 基础 3：JSON 往返

把 record = {"date": "2026-08-06", "category": "餐饮", "amount": 35.5} 用 dumps/loads 往返一次并验证相等。


In [ ]:
# 请在下方填写代码
import json

record = {"date": "2026-08-06", "category": "餐饮", "amount": 35.5}
# TODO：请在下方完成 —— 基础 3：JSON 往返 把 record = {"date": "2026-08-06", "category": "


In [ ]:
import json

record = {"date": "2026-08-06", "category": "餐饮", "amount": 35.5}
json_text = json.dumps(record, ensure_ascii=False)
restored = json.loads(json_text)
print(json_text)
print(restored == record)


### 提高 1：save_data 与 load_data

实现 save_data（自动创建目录）与 load_data（文件不存在返回空列表），在临时目录中验证：首次读取为空、保存后读取一致。


In [ ]:
# 请在下方填写代码
import json
from pathlib import Path
from tempfile import TemporaryDirectory

# TODO：请在下方完成 —— 提高 1：save_data 与 load_data 实现 save_data（自动创建目录）与 load_data（文


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory


def load_data(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def save_data(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)


with TemporaryDirectory() as folder:
    path = Path(folder) / "data" / "records.json"
    empty = load_data(path)
    records = [{"type": "支出", "category": "餐饮", "amount": 35.5}]
    save_data(records, path)
    restored = load_data(path)
    print(empty, restored)
_ok = empty == [] and restored == records
print("诊断：", "通过" if _ok else "检查首次读取、目录创建或 JSON")


### 提高 2：文件写入检查

在临时目录写入两笔记录（含收入与支出），读取后按类型统计金额合计。


In [ ]:
# 请在下方填写代码
import json
from pathlib import Path
from tempfile import TemporaryDirectory

records = [
    {"type": "收入", "category": "工资", "amount": 5000.0},
    {"type": "支出", "category": "餐饮", "amount": 35.5},
]
# TODO：请在下方完成 —— 提高 2：文件写入检查 在临时目录写入两笔记录（含收入与支出），读取后按类型统计金额合计。


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

records = [
    {"type": "收入", "category": "工资", "amount": 5000.0},
    {"type": "支出", "category": "餐饮", "amount": 35.5},
]
with TemporaryDirectory() as folder:
    path = Path(folder) / "records.json"
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)
    with path.open(encoding="utf-8") as file:
        loaded = json.load(file)
    income = sum(item["amount"] for item in loaded if item["type"] == "收入")
    expense = sum(item["amount"] for item in loaded if item["type"] == "支出")
    print(income, expense)


### 挑战 1：完整持久化闭环

实现 load_data 与 save_data，验证：首次读取空列表 → 保存两笔记录 → 重新读取一致 → 重复保存不丢失。


In [ ]:
# 请在下方填写代码
import json
from pathlib import Path
from tempfile import TemporaryDirectory

# TODO：请在下方完成 —— 挑战 1：完整持久化闭环 实现 load_data 与 save_data，验证：首次读取空列表 → 保存两笔记录 →


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory


def load_data(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as file:
        data = json.load(file)
    return data if isinstance(data, list) else []


def save_data(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)


with TemporaryDirectory() as folder:
    path = Path(folder) / "data" / "records.json"
    empty = load_data(path)
    records = [{"type": "支出", "category": "餐饮", "amount": 35.5}]
    save_data(records, path)
    records.append({"type": "支出", "category": "交通", "amount": 18.0})
    save_data(records, path)
    restored = load_data(path)
    print(empty, restored)
_ok = empty == [] and len(restored) == 2
print("诊断：", "通过" if _ok else "检查持久化闭环")


## 12.9 小结

### 知识要点回顾

| 知识点 | 要点 |
| --- | --- |
| Path | / 组合路径；name/stem/suffix/parent |
| 目录 | mkdir(parents=True, exist_ok=True) |
| with | 自动关闭文件；异常安全 |
| 模式 | w 覆盖 / a 追加 / r 读取 |
| 编码 | 中文必须 encoding="utf-8" |
| JSON 文件 | dump / load |
| JSON 字符串 | dumps / loads（带 s） |
| 持久化 | save_data / load_data 闭环 |

### 自测清单

- [ ] 能用 Path 构造与解析路径；
- [ ] 能用 with 读写文本文件；
- [ ] 能区分 w/a/r 三种模式；
- [ ] 能用 dump/load 保存读取 JSON；
- [ ] 能区分 dump/load 与 dumps/loads；
- [ ] 能封装 load_data/save_data 并验证闭环。




## 12.10 拓展作业

### 必做作业

**作业 1（账本持久化）**：实现 load_data/save_data，保存 3 笔记录后重启式读取（重新加载），输出记录数与金额合计。

**作业 2（备份机制）**：用 with_suffix 生成 records.backup.json，把当前记录备份一份，并验证两个文件都存在。

### 选做拓展

研究 pathlib 的 resolve() 与 absolute()：打印相对路径与绝对路径，讨论程序发布后路径基准应该怎么定。

### 下章预习

第 13 章《异常处理、调试与基础测试》将学习：异常类型、try/except/else/finally、raise、自定义异常与表驱动测试。预习时思考：load_data 读取损坏的 JSON 文件时应该怎么办？




## 教学实验：边界条件

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
orders = [
    {"order_id": "A01", "amount": 280},
    {"order_id": "A02", "amount": 300},
    {"order_id": "A03", "amount": 520},
]
threshold = 300
for order in orders:
    label = "达到门槛" if order["amount"] >= threshold else "未达到门槛"
    print(order["order_id"], order["amount"], label)


### 第一个结果怎么读

这里的重点不是记住 `if`，而是观察 `>=` 如何处理刚好等于 300 的记录。边界条件必须和业务规则保持一致。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
threshold = 500
for order in orders:
    label = "重点关注" if order["amount"] >= threshold else "普通订单"
    print(order["order_id"], "->", label)
print("重点订单数：", sum(order["amount"] >= threshold for order in orders))


### 第二个结果怎么读

只把门槛从 300 改成 500，再比较标签和数量变化。这个实验训练的是“改一个输入，解释一个输出”。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：类型转换失败怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
amount_text = "128.5"
try:
    amount = int(amount_text)
except ValueError as error:
    print("第一次转换失败：", type(error).__name__)
    amount = float(amount_text)
print("可以继续使用的金额：", amount)


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先读错误类型，再决定修复方法。这里不是盲目忽略错误，而是明确知道整数转换不适合带小数的文本。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。
